# Step 2: feature engineering
**BT4222 Group 5 · Impression-Aware Re-ranking of Search Results on JD.com**

Builds one feature row per impression (user, candidate) for every slate. Every feature has one of two roles.

**Model features** are the set step 4 trains on. Each belongs to a hypothesis group written down before any result was seen: what the features should capture, and which finding motivated them. Section 5 gives every feature univariate evidence. Step 4 then tests each group by removing it and measuring validation NDCG, and keeps a group only if removing it hurts. The set is fixed in `model_features.json`, so nothing joins it ad hoc later.

**Comparison features** are the interim plan exactly as submitted (the `int_` features and pooled popularity). Step 3 trains them as a baseline and section 5 sets them against the model features. They never enter the final model.

The interim feedback shapes the model set. The professor asked us to convert the history gaps into days before the query and watch the mix of clicks, carts and orders further back. Step 1 found each type is recorded for a different period (`history_windows.json`), so no model feature compares types outside the period where all of them are recorded.

| | |
|---|---|
| **Inputs** | `processed/parquet/` and `processed/artifacts/history_windows.json` from step 1 |
| **Outputs** | `processed/artifacts/features/features_*.parquet` (one part per step-1 part), `model_features.json`, `feature_dictionary.csv` (role, hypothesis and evidence per feature), `user_features.parquet`, `item_features.parquet`, `folds.parquet` |
| **Runtime** | CPU, standard RAM. About 7 minutes for Run all; pass 1 (3.1) takes about 3 of them and pass 2 (4.2) under 2. Set `CFG["use_names"] = False` if Colab runs out of memory (drops group H8 and the interim name similarity) |
| **Next** | `step3_baselines.ipynb` trains the comparison set; `step4_advanced_models.ipynb` reads `model_features.json`; `step5_diagnostics.ipynb` reuses the feature rows |

**How to run.** Run step 1 first (with outputs saved), then Runtime → Run all here.

## What changed from the interim plan, and why

| Interim feature (decision) | Here | Why |
|---|---|---|
| Purchase share of history (kept) | Comparison `int_purchase_share`; model H1 `u_orders_per_cart_w`, `u_carts_per_click_w` | Pooled, it divides a year of orders by days of clicks and months of carts, so it rises for anyone quiet lately |
| Order count (dropped) | Comparison `int_order_count`; model H1 `u_n_order_30d`, `u_n_order_w` | Orders cover one consistent period, so counts of them are clean |
| Click share, cart share (dropped) | Comparison only | They mostly record where activity fell relative to the cutoffs |
| Search dependence (kept) | Comparison only | Moves with the type mix; no window-aware version has its own hypothesis |
| Recency (kept) | Comparison `int_recency_log`; model H1 `u_days_since_click`, `u_days_since_order` | "Last event" means a click for active users and an order for quiet ones |
| History length (kept for its interaction) | Comparison only | It adds up periods of different lengths |
| Burstiness (dropped) | Comparison only | Older gaps are long partly because clicks and carts were never kept |
| 27-cell segmentation | Comparison `int_segment27` (cut points from training users) | Reproduced so the report can compare |
| Shop, brand, category match over all types | Comparison `int_*_match`; model H2 split by source | Pooled, it mixes the last few days of clicks with a year of orders |
| Similarity against the user's history | Comparison `int_name_sim`; model H8 split by source | Same pooling question |
| Impressions across all slates | `i_exposure_n`, in both sets | Label-free |
| Interaction rate across slates | `i_eng_rate_oof` in both sets; model H6 adds the purchase rate | The original reads the labels it predicts (75.2% of items appear in one slate, so for them it *is* the label). Now out of fold, training users only, and empty when no other training user saw the item |
| Shown before without acting | Dropped | No data source: history holds no impressions and each user has one slate |
| Query-term overlap (tested, dropped) | Still dropped | Weak in the interim because retrieval already matched every candidate to the query |

Three comparison features are our reconstruction rather than the interim's exact definition: burstiness (median seconds between consecutive events), name similarity (share of the candidate's name terms found in the history, not a thresholded cosine combined with shop and brand), and the 27-cell cut points (training users only, to avoid leakage).

:## Model features: one hypothesis per group

| Group | Hypothesis | Features | Comes from |
|---|---|---|---|
| **H1** Activity inside its window | Clicks, carts and orders describe current intent and buying habit only when counted inside the period each type is recorded, and compared only where both types exist | `u_n_click_w`, `u_carts_per_click_w`, `u_orders_per_cart_w`, `u_n_order_30d`, `u_n_order_w`, `u_days_since_click`, `u_days_since_order` | Step 1 timeline; interim feedback |
| **H2** Where a match comes from | A shop, brand or category the user recently clicked signals intent, one they carted signals consideration, one they ordered signals loyalty; pooled, these mix | `ui_days_{shop,brand,cate4}_{click,cart,order}` (9) | Interim Table 11 used all types together |
| **H3** The item itself | Users return to items they recently clicked and buy items left in the cart; a recent order of the same item may mean the need is already met | `ui_item_days_click`, `ui_item_days_cart`, `ui_item_days_order`, `ui_item_cart_no_order` | Missing from the interim; usable once event ages are known |
| **H4** The query as a key into history | Past behaviour under a search like the current one matters more than the rest of the history | `q_seen_before`, `ui_item_days_related_query`, `ui_cate4_days_related_query` | "Distinct roles of your data fields" in the feedback |
| **H5** Popularity by type | What many users recently clicked differs from what many users bought over a year; both describe an item's appeal | `i_pop_click`, `i_pop_cart`, `i_pop_order`, `i_exposure_n` | Types cover different periods (step 1); interim exposure feature |
| **H6** Item conversion rate | Some items convert better than others, measurable from other users' outcomes without seeing the row's own label | `i_eng_rate_oof`, `i_pur_rate_oof`, `i_label_imps_oof` | Interim interaction rate, leak fixed; empty when no other training user saw the item (change 2) |
| **H7** Standing within the slate | A re-ranker compares candidates within one slate, so popularity relative to the slate, and how much of the slate the shop fills, matter more than absolute counts | `s_pop_order_pct`, `s_pop_click_pct`, `s_shop_share` | The task is within-slate ordering |
| **H8** Name overlap by source | Name terms shared with recently clicked or carted items signal current intent; shared with ordered items, a repeat or a substitute | `ui_name_cov_recent`, `ui_name_cov_order` | Interim similarity, split like H2 |

**How the hypotheses are tested.** Section 5.3 gives every feature a *within-slate AUC*: across training slates, the chance that a candidate the user engaged with scores higher on the feature than one they ignored. 0.5 means no ranking signal on its own. Features that are the same for every candidate in a slate (all of H1, and `q_seen_before`) are exactly 0.5 by construction: they can only help by changing how much the item features count, which only the step-4 ablation can show. That ablation decides which groups stay; this notebook does not drop anything based on its own results.

## Considered and not used

| Feature | Why not |
|---|---|
| Query length, slate size | Same for every candidate in a slate, with no mechanism by which they would change the order |
| Cold-item flag | Identical to pooled popularity being zero |
| Missing-metadata flag | 0.68% of candidates; missing attributes already show up as empty matches |
| Search share per type, distinct days with clicks | No clear mechanism linking them to which item gets chosen |
| Days since the last event | The interim recency again (kept in the comparison set) |
| Number of related past searches | Same for the whole slate; H4's item-level features carry the useful part |
| Exposure percentile within the slate | No mechanism beyond raw exposure (H5) |
| Carts in window, orders in 90 days, days since last cart | Overlap with H1 and H3 without a separate hypothesis |

## Changes after the first full run

The first full run (all 173,831 users) is the baseline for this log. Every change made after seeing its output is listed here, with what triggered it, so the report can separate what was planned from what was learned.

| # | Change | Triggered by | Why it is a correction, not tuning | Affects |
|---|---|---|---|---|
| 1 | Empty search terms are read as "no term" | A crash on the real data: some recorded searches have empty query text, or doubled separators | A parsing bug. The synthetic test data never contained empty terms | Nothing in the design; the run could complete |
| 2 | H6 rates are **empty** when no other training user saw the item, instead of taking the overall rate | Section 5.3 of the first run: purchase rate within-slate AUC 0.331, engagement rate 0.546 | Filling unseen items with the average made "no data" look like "average". Items other users saw mostly have zero purchases, so their smoothed rate fell below the average, and the feature ranked well-exposed items below unseen ones. Empty lets the model tell "no data" from "low rate". The hypothesis, the smoothing and the out-of-fold scheme are unchanged | `i_eng_rate_oof`, `i_pur_rate_oof` (the first is also in the interim baseline) |
| 3 | Section 5.1 shows yes/no features as "without vs with" | Reading 5.1: `q_seen_before` showed 1.00x to 1.02x across "quartiles" | Quartiles of a yes/no feature split tied users arbitrarily | The evidence table only |

**Not changed after the run.** No feature was added to or removed from the model set, and no hypothesis was reworded. H3's clause that a recent order may mean the need is already met stays as written; the first run contradicts it (past orders of the same item carried a 67x purchase lift), and the report states that as a refuted prediction.

**Declared exploratory variant.** Pooled popularity (`i_pop_any`) had the highest within-slate AUC in the first run (0.777), above every per-type popularity feature in H5. Because that was found by looking, it is *not* added to the model set. Step 4 tests "model set + pooled popularity" on validation users and reports it separately, labelled as found after looking.

**Frozen.** The model set is version 1.0 in `model_features.json`. Step 4 decides which groups stay using validation users only; test users are scored once, at the end.

## 0. Setup

In [ ]:
import os, glob, json, time, random
from contextlib import contextmanager
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from IPython.display import display, HTML

NOTEBOOK = "step2_feature_engineering"
try:                                   # Colab: mount Drive. Elsewhere: set JD_BASE instead
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

BASE = os.environ.get("JD_BASE", "/content/drive/MyDrive/JDsearch_rawData")   # shared team folder
assert os.path.isdir(BASE), f"{BASE} not found: add a shortcut to JDsearch_rawData in My Drive"
PROC = os.path.join(BASE, "processed")
PATHS = {"parquet": os.path.join(PROC, "parquet"),
         "artifacts": os.path.join(PROC, "artifacts"),
         "features": os.path.join(PROC, "artifacts", "features"),
         "figures": os.path.join(PROC, "results", "figures")}
for key in ("artifacts", "features", "figures"):
    os.makedirs(PATHS[key], exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
print("reading:", PATHS["parquet"])
print("writing:", PATHS["features"])

Mounted at /content/drive
reading: /content/drive/MyDrive/JDsearch_rawData/processed/parquet
writing: /content/drive/MyDrive/JDsearch_rawData/processed/artifacts/features


In [ ]:
# 0.1 Configuration
CFG = {
    "seed": SEED,
    "split": {"train": 0.8, "val": 0.1, "test": 0.1},   # same user split as the template, created once
    "oof_folds": 5,              # folds over training users for label-based item rates
    "prior_strength": 20,        # smoothing: the global rate counts as this many impressions
    "use_names": True,           # name-term features (largest memory use); False if Colab runs out of RAM
    "auc_users": 30_000,         # training users whose slates give the within-slate AUC in section 5.3
}
with open(os.path.join(PATHS["artifacts"], "history_windows.json")) as f:
    WIN = json.load(f)["windows_days"]
W_CLICK, W_CART, W_ORD = float(WIN["CLICK"]), float(WIN["CART"]), float(WIN["ORD"])
CLICK, CART, ORD, FLW = 1, 2, 3, 4
TERM_SEP = "\x18"
T_KEY = 2 ** 31                  # ids fit in int32, so (owner * T_KEY + id) is a unique int64 key
print(f"history windows from step 1: clicks {W_CLICK:g} days, carts {W_CART:g} days, orders {W_ORD:g} days")

# Model feature set: one hypothesis per group, written before any result was seen (table at the top)
GROUPS = {
    "H1 activity inside its window": ["u_n_click_w", "u_carts_per_click_w", "u_orders_per_cart_w", "u_n_order_30d",
                                      "u_n_order_w", "u_days_since_click", "u_days_since_order"],
    "H2 where a match comes from": [f"ui_days_{a}_{s}" for a in ("shop", "brand", "cate4") for s in ("click", "cart", "order")],
    "H3 the item itself": ["ui_item_days_click", "ui_item_days_cart", "ui_item_days_order", "ui_item_cart_no_order"],
    "H4 query as a key into history": ["q_seen_before", "ui_item_days_related_query", "ui_cate4_days_related_query"],
    "H5 popularity by type": ["i_pop_click", "i_pop_cart", "i_pop_order", "i_exposure_n"],
    "H6 item conversion rate": ["i_eng_rate_oof", "i_pur_rate_oof", "i_label_imps_oof"],
    "H7 standing within the slate": ["s_pop_order_pct", "s_pop_click_pct", "s_shop_share"],
    "H8 name overlap by source": ["ui_name_cov_recent", "ui_name_cov_order"],
}
HYPOTHESIS = {
    "H1 activity inside its window": "Activity describes intent and habit only inside the period each type is recorded",
    "H2 where a match comes from": "Recent clicks signal intent, carts consideration, orders loyalty; pooled, these mix",
    "H3 the item itself": "Users return to recently clicked items and buy items left in the cart",
    "H4 query as a key into history": "Past behaviour under a related search matters more than the rest of the history",
    "H5 popularity by type": "Recent click popularity and long-run order popularity describe different kinds of appeal",
    "H6 item conversion rate": "Some items convert better, measurable from other users' outcomes",
    "H7 standing within the slate": "A re-ranker compares within a slate, so relative standing matters",
    "H8 name overlap by source": "Name overlap with recent items signals intent; with ordered items, repeats or substitutes",
}
# Comparison set: the interim plan as submitted. The interim baseline is what its re-ranker would have read.
INTERIM_BASELINE = ["int_purchase_share", "int_search_dependence", "int_recency_log", "int_history_length",
                    "int_segment27", "int_shop_match", "int_brand_match", "int_cate4_match", "int_name_sim",
                    "i_exposure_n", "i_eng_rate_oof"]
COMPARISON = INTERIM_BASELINE + ["int_order_count", "int_click_share", "int_cart_share", "int_burstiness_log", "i_pop_any"]
if not CFG["use_names"]:
    GROUPS.pop("H8 name overlap by source")
    INTERIM_BASELINE.remove("int_name_sim")
    COMPARISON.remove("int_name_sim")
MODEL = [f for feats in GROUPS.values() for f in feats]
GROUP_OF = {f: g for g, feats in GROUPS.items() for f in feats}
print(f"model set: {len(MODEL)} features in {len(GROUPS)} groups | comparison set: {len(COMPARISON)} features")

history windows from step 1: clicks 7 days, carts 90 days, orders 365 days
model set: 35 features in 8 groups | comparison set: 16 features


## 1. Shared helpers

In [ ]:
def parts(prefix):
    """Step-1 parquet parts. Refuses Drive's duplicate copies (e.g. 'name (1).parquet')."""
    files = sorted(glob.glob(os.path.join(PATHS["parquet"], f"{prefix}_*.parquet")))
    assert files, f"no {prefix}_*.parquet in {PATHS['parquet']}: run step 1 first"
    copies = [os.path.basename(f) for f in files if " (" in os.path.basename(f)]
    assert not copies, f"duplicate copies in {PATHS['parquet']}: {copies[:3]}. Delete them or re-run step 1"
    return files

@contextmanager
def timer(label):
    start = time.time()
    yield
    print(f"[{label}] {time.time() - start:,.1f}s")

def show_result(title, obj=None, note=""):
    """Red RESULT banner, then the table, so key outputs stand out in the saved notebook."""
    extra = f'<div style="font-size:90%;margin-top:3px">{note}</div>' if note else ""
    display(HTML('<div style="border-left:6px solid #c0392b;background:#fbeaea;color:#1f1f1f;'
                 'padding:8px 12px;margin:12px 0 4px 0;font-family:Arial,sans-serif">'
                 f"<b>RESULT</b> &nbsp;|&nbsp; {title}{extra}</div>"))
    if obj is not None:
        display(obj)

def to_int(col):
    """Arrow column of ids (integer or text) -> int64 numpy array, -1 where missing."""
    if pa.types.is_integer(col.type):
        return pc.fill_null(col, -1).to_numpy().astype(np.int64)
    try:
        return pc.fill_null(pc.cast(col, pa.int64()), -1).to_numpy()
    except (pa.ArrowInvalid, pa.ArrowNotImplementedError):
        return pd.to_numeric(col.to_pandas(), errors="coerce").fillna(-1).astype(np.int64).to_numpy()

def terms_to_int(words):
    """Arrow array of term-id strings -> int64 numpy, with -1 for empty, missing or non-numeric terms.
    Some recorded searches have empty query text, or doubled separators that leave empty terms."""
    words = pc.fill_null(words, "")
    words = pc.if_else(pc.equal(words, ""), "-1", words)
    try:
        return pc.cast(words, pa.int64()).to_numpy()
    except pa.ArrowInvalid:
        return pd.to_numeric(pd.Series(words.to_pylist()), errors="coerce").fillna(-1).astype(np.int64).to_numpy()

def flat(t, col):
    """List column -> (all values as one numpy array, list length per user)."""
    return pc.list_flatten(t[col]).to_numpy(), pc.list_value_length(t[col]).to_numpy()

def get_split():
    """Fixed user-level split shared by every notebook (identical to the template's)."""
    path = os.path.join(PATHS["artifacts"], "split_users.parquet")
    if not os.path.exists(path):
        uid = np.concatenate([pq.read_table(f, columns=["user_id"])["user_id"].to_numpy()
                              for f in parts("user_behavior")]).astype(np.int32)
        r = np.random.default_rng(CFG["seed"]).random(len(uid))
        cut_train, cut_val = CFG["split"]["train"], CFG["split"]["train"] + CFG["split"]["val"]
        split = np.where(r < cut_train, "train", np.where(r < cut_val, "val", "test"))
        pd.DataFrame({"user_id": uid, "split": split}).to_parquet(path, index=False)
        print(f"created {path}")
    return pd.read_parquet(path)

def history_frame(t):
    """One row per history event of the users in an Arrow table from step 1."""
    wid, n = flat(t, "history_wids")
    first = np.zeros(len(wid), bool)
    first[np.cumsum(n) - n] = True
    return pd.DataFrame({
        "user_id": np.repeat(t["user_id"].to_numpy().astype(np.int32), n),
        "wid": wid.astype(np.int64),
        "type": flat(t, "history_types")[0].astype(np.int8),
        "age": flat(t, "history_age_days")[0].astype(np.float32),
        "has_query": np.asarray(pc.list_flatten(t["history_has_query"])),
        "gap": flat(t, "history_gaps")[0],
        "first": first,
    })

def query_links(t, h_ul):
    """Link each past event to the current search.
    Returns (same_query, related) event masks and the (user, term) keys of the current query.
    same_query: the event's search used exactly the current query.
    related: the event's search shares at least one term with the current query."""
    q_terms, q_n = flat(t, "query_terms")
    q_keys = np.unique(np.repeat(np.arange(t.num_rows), q_n).astype(np.int64) * T_KEY + q_terms)
    current = np.array([TERM_SEP.join(map(str, q)) for q in t["query_terms"].to_pylist()], dtype=object)
    ev_q = pa.concat_arrays(pc.list_flatten(pc.split_pattern(t["history_qry_packed"], "_")).chunks)
    same = np.asarray(ev_q.to_pandas(), dtype=object) == current[h_ul]
    ev_lists = pc.split_pattern(ev_q, TERM_SEP)
    ev_idx = pc.list_parent_indices(ev_lists).to_numpy()
    ev_terms = terms_to_int(pc.list_flatten(ev_lists))
    ok = ev_terms >= 0                                   # -1: no search behind the event, or an empty term
    hit = np.isin(h_ul[ev_idx[ok]].astype(np.int64) * T_KEY + ev_terms[ok], q_keys)
    related = np.zeros(len(h_ul), bool)
    related[ev_idx[ok][hit]] = True
    return same, related, q_keys

def min_age_lookup(owner, ids, age, q_owner, q_ids):
    """For each (q_owner, q_ids) pair: the smallest age among events with the same (owner, id), NaN if none."""
    if len(owner) == 0:
        return np.full(len(q_owner), np.nan, dtype=np.float32)
    s = pd.Series(age).groupby(owner.astype(np.int64) * T_KEY + ids).min()
    return s.reindex(q_owner.astype(np.int64) * T_KEY + q_ids).to_numpy().astype(np.float32)

## 2. Shared inputs: split, folds and product attributes

**Folds.** Label-based item statistics must never see the row they describe. Training users are split into 5 folds; a training row's item rate comes from the other 4 folds, and validation and test rows use all training users. The folds are saved so later steps can reuse them.

**Checked in step 5.** Because a training row loses its own fold while validation and test rows use all five, the training impressions behind an item's rates (`i_label_imps_oof`) are about 1.42 times larger for test rows than for training rows (1.01 against the expected 0.8 × (N − 1)). Step 5 section 7 puts the test counts on the training scale and re-scores the final model: NDCG@10 changes by −0.0001 [−0.0004, +0.0002]. The gap does not bias the results, so the folds are kept as they are.

**Name terms.** Product names are sequences of anonymised term ids. They are loaded once into flat arrays (about 1.4 GB for the full data) so that name overlap can be computed with array operations instead of Python loops.

In [ ]:
# 2.1 Split and out-of-fold assignment
split = get_split()
train_ids = np.sort(split.loc[split["split"] == "train", "user_id"].to_numpy())
folds = pd.DataFrame({"user_id": train_ids,
                      "fold": np.random.default_rng(SEED + 1).integers(0, CFG["oof_folds"], len(train_ids))})
folds.to_parquet(os.path.join(PATHS["artifacts"], "folds.parquet"), index=False)
SPLIT_OF = split.set_index("user_id")["split"]
FOLD_OF = pd.Series(-1, index=split["user_id"].to_numpy(), dtype=np.int8)
FOLD_OF.loc[train_ids] = folds["fold"].to_numpy().astype(np.int8)
display(split["split"].value_counts().to_frame("users"))

# 2.2 Product attributes, sorted by wid for fast lookups
with timer("product attributes"):
    t = pa.concat_tables([pq.read_table(f, columns=["wid", "brand_id", "shop_id", "cate_id_4"])
                          for f in parts("product_meta")])
    cols = {c: to_int(t[c]) for c in ("wid", "brand_id", "shop_id", "cate_id_4")}
    order = np.argsort(cols["wid"], kind="stable")
    META = {c: v[order] for c, v in cols.items()}
    del t, cols, order
assert max(META[c].max() for c in META) < T_KEY, "an id does not fit the int64 key scheme"
print(f"{len(META['wid']):,} products with attributes")

def attrs(wids):
    """brand, shop and finest category for each wid (-1 when the product has no metadata)."""
    pos = np.clip(np.searchsorted(META["wid"], wids), 0, len(META["wid"]) - 1)
    hit = META["wid"][pos] == wids
    return {c: np.where(hit, META[c][pos], -1) for c in ("brand_id", "shop_id", "cate_id_4")}, hit

,users
split,
train,139075
test,17505
val,17251


[product attributes] 34.0s
12,141,247 products with attributes


In [ ]:
# 2.3 Product name terms (skipped when CFG["use_names"] is False)
if CFG["use_names"]:
    with timer("name terms"):
        w_list, len_list, term_list = [], [], []
        for f in parts("product_meta"):
            t = pq.read_table(f, columns=["wid", "name"])
            lists = pc.split_pattern(pc.fill_null(t["name"], ""), TERM_SEP)
            words = pc.list_flatten(lists)
            vals = terms_to_int(words)
            assert vals.max() < T_KEY, "a name term id does not fit int32"
            w_list.append(to_int(t["wid"]))
            len_list.append(pc.list_value_length(lists).to_numpy().astype(np.int64))
            term_list.append(vals.astype(np.int32))
            del t, lists, words, vals
        NAME_WID = np.concatenate(w_list)
        NAME_LEN = np.concatenate(len_list)
        NAME_TERMS = np.concatenate(term_list)
        del w_list, len_list, term_list
        NAME_OFF = np.cumsum(NAME_LEN) - NAME_LEN
        NAME_ROW_S = np.argsort(NAME_WID, kind="stable")
        NAME_WID_S = NAME_WID[NAME_ROW_S]
    print(f"{len(NAME_TERMS):,} name terms for {len(NAME_WID):,} products")

def gather_terms(wids):
    """Name terms of each wid as (index into wids, term) pairs; products without metadata give none."""
    pos = np.clip(np.searchsorted(NAME_WID_S, wids), 0, len(NAME_WID_S) - 1)
    hit = NAME_WID_S[pos] == wids
    rows = NAME_ROW_S[pos[hit]]
    lengths, starts = NAME_LEN[rows], NAME_OFF[rows]
    idx = np.arange(int(lengths.sum())) + np.repeat(starts - (np.cumsum(lengths) - lengths), lengths)
    owner, terms = np.repeat(np.flatnonzero(hit), lengths), NAME_TERMS[idx]
    keep = terms >= 0
    return owner[keep], terms[keep].astype(np.int64)

[name terms] 70.1s
338,901,750 name terms for 12,141,247 products


## 3. Pass 1: user features and item statistics

One pass over the step-1 parts computes every user-level feature and the item statistics that need all users at once.

**User features.** The comparison versions use the whole history pooled, exactly as submitted. The H1 versions count each type only inside its recorded period and compare types only inside the shortest period covering both: carts per click inside the click window, orders per cart inside the cart window. `q_seen_before` (H4) marks users who ran exactly this query before.

**Item statistics.** Popularity counts distinct users whose history holds the item: pooled over everything for the comparison set, per type inside its window for H5. Exposure is the number of slates an item appears in. Label counts come from training users only, per fold, for H6.

In [ ]:
# 3.1 Pass 1 over the step-1 parts
def user_features(t, h):
    uid = t["user_id"].to_numpy().astype(np.int32)
    out = pd.DataFrame(index=pd.Index(uid, name="user_id"))
    by = h["user_id"]
    n = h.groupby("user_id").size()
    cnt = h.groupby(["user_id", "type"]).size().unstack(fill_value=0).reindex(columns=[1, 2, 3, 4], fill_value=0)
    gap_to_q = t["gap_to_test_query"].to_numpy().astype(float)

    # comparison set: interim definitions, whole history pooled, as submitted
    out["int_history_length"] = n
    out["int_order_count"] = cnt[ORD]
    out["int_purchase_share"] = cnt[ORD] / n
    out["int_click_share"] = cnt[CLICK] / n
    out["int_cart_share"] = cnt[CART] / n
    out["int_search_dependence"] = h.groupby("user_id")["has_query"].mean()
    out["int_recency_log"] = np.log1p(gap_to_q)
    out["int_burstiness_log"] = np.log1p(h.loc[~h["first"]].groupby("user_id")["gap"].median())

    # H1: each type only inside the period where it is recorded; ratios only where both types exist
    click_w = (h["type"] == CLICK) & (h["age"] < W_CLICK)
    cart_w = (h["type"] == CART) & (h["age"] < W_CART)
    ord_w = (h["type"] == ORD) & (h["age"] < W_ORD)
    n_click, n_cart = click_w.groupby(by).sum(), cart_w.groupby(by).sum()
    out["u_n_click_w"] = n_click
    out["u_carts_per_click_w"] = ((h["type"] == CART) & (h["age"] < W_CLICK)).groupby(by).sum() / n_click.replace(0, np.nan)
    out["u_orders_per_cart_w"] = ((h["type"] == ORD) & (h["age"] < W_CART)).groupby(by).sum() / n_cart.replace(0, np.nan)
    out["u_n_order_30d"] = ((h["type"] == ORD) & (h["age"] < min(30.0, W_ORD))).groupby(by).sum()
    out["u_n_order_w"] = ord_w.groupby(by).sum()
    out["u_days_since_click"] = h.loc[click_w].groupby("user_id")["age"].min()
    out["u_days_since_order"] = h.loc[ord_w].groupby("user_id")["age"].min()
    return out

user_parts, pop_any_parts, pop_type_parts, expo_parts, label_parts = [], [], [], [], []
with timer("pass 1"):
    for f in parts("user_behavior"):
        t = pq.read_table(f, columns=["user_id", "query_terms", "candidate_wids", "candidate_labels", "history_wids",
                                      "history_types", "history_gaps", "gap_to_test_query", "history_age_days",
                                      "history_has_query", "history_qry_packed"])
        uid = t["user_id"].to_numpy().astype(np.int32)
        h = history_frame(t)
        h_ul = np.searchsorted(uid, h["user_id"].to_numpy())
        u = user_features(t, h)
        same, _, _ = query_links(t, h_ul)
        u["q_seen_before"] = (np.bincount(h_ul, weights=same, minlength=len(uid)) > 0).astype(float)

        # outcome counts per user, used only by the evidence tables in section 5
        wids, n_c = flat(t, "candidate_wids")
        labs = flat(t, "candidate_labels")[0]
        starts = np.cumsum(n_c) - n_c
        u["n_imp"] = n_c
        u["n_eng"] = np.add.reduceat((labs > 0).astype(np.int32), starts)
        u["n_pur"] = np.add.reduceat((labs == 3).astype(np.int32), starts)
        user_parts.append(u)

        # item popularity: pooled (comparison) and per type inside its window (H5)
        pop_any_parts.append(h.drop_duplicates(["user_id", "wid"])["wid"].value_counts())
        in_win = (((h["type"] == CLICK) & (h["age"] < W_CLICK)) | ((h["type"] == CART) & (h["age"] < W_CART))
                  | ((h["type"] == ORD) & (h["age"] < W_ORD)))
        pop_type_parts.append(h.loc[in_win].drop_duplicates(["user_id", "wid", "type"]).groupby(["wid", "type"]).size())
        expo_parts.append(pd.Series(wids).value_counts())

        # label counts from training users only, per fold (H6)
        fold = FOLD_OF.reindex(np.repeat(uid, n_c)).to_numpy()
        tr = fold >= 0
        label_parts.append(pd.DataFrame({"wid": wids[tr], "fold": fold[tr], "imp": 1,
                                         "eng": (labs[tr] > 0).astype(np.int32),
                                         "pur": (labs[tr] == 3).astype(np.int32)})
                           .groupby(["wid", "fold"]).sum())
        print(f"  {os.path.basename(f)}: {len(uid):,} users, {len(h):,} events")

USERS = pd.concat(user_parts)
pop_type = pd.concat(pop_type_parts).groupby(level=[0, 1]).sum().unstack(fill_value=0)
ITEMS = pd.DataFrame({
    "i_pop_any": pd.concat(pop_any_parts).groupby(level=0).sum(),
    "i_pop_click": pop_type.get(CLICK), "i_pop_cart": pop_type.get(CART), "i_pop_order": pop_type.get(ORD),
    "i_exposure_n": pd.concat(expo_parts).groupby(level=0).sum(),
}).fillna(0).astype(np.float32)
LAB_FOLD = pd.concat(label_parts).groupby(level=[0, 1]).sum()
LAB_TOT = LAB_FOLD.groupby(level=0).sum()
P_ENG = LAB_TOT["eng"].sum() / LAB_TOT["imp"].sum()
P_PUR = LAB_TOT["pur"].sum() / LAB_TOT["imp"].sum()
LAB_FOLD.index = LAB_FOLD.index.get_level_values(0).astype(np.int64) * 16 + LAB_FOLD.index.get_level_values(1)
del user_parts, pop_any_parts, pop_type_parts, expo_parts, label_parts, pop_type
print(f"users {len(USERS):,} | items with any statistic {len(ITEMS):,} | "
      f"training engagement rate {P_ENG:.3%}, purchase rate {P_PUR:.3%}")

  user_behavior_00.parquet: 20,000 users, 3,106,083 events
  user_behavior_01.parquet: 20,000 users, 3,036,952 events
  user_behavior_02.parquet: 20,000 users, 3,097,281 events
  user_behavior_03.parquet: 20,000 users, 3,015,572 events
  user_behavior_04.parquet: 20,000 users, 3,036,329 events
  user_behavior_05.parquet: 20,000 users, 3,112,535 events
  user_behavior_06.parquet: 20,000 users, 3,054,807 events
  user_behavior_07.parquet: 20,000 users, 3,115,865 events
  user_behavior_08.parquet: 13,831 users, 2,091,836 events
[pass 1] 106.5s
users 173,831 | items with any statistic 12,319,978 | training engagement rate 2.075%, purchase rate 0.134%


In [ ]:
# 3.2 The interim 27-cell segmentation, with cut points from training users only
SEG = ["int_purchase_share", "int_recency_log", "int_search_dependence"]
train_users = USERS.loc[USERS.index.isin(train_ids)]
cuts = {f: train_users[f].quantile([1 / 3, 2 / 3]).to_numpy() for f in SEG}
USERS["int_segment27"] = sum(np.digitize(USERS[f], cuts[f]) * w for f, w in zip(SEG, (9, 3, 1))).astype(float)

cell = USERS.loc[USERS.index.isin(train_ids)].groupby("int_segment27")[["n_pur", "n_imp"]].sum()
cell_rate = cell["n_pur"] / cell["n_imp"]
show_result("Interim 27-cell segmentation, reproduced on training users",
            pd.DataFrame({"cells": [len(cell)], "lowest purchase rate": [f"{cell_rate.min():.4%}"],
                          "highest purchase rate": [f"{cell_rate.max():.4%}"],
                          "spread": [f"{cell_rate.max() / cell_rate.min():.1f}x"]}),
            note="The interim reported 27 cells spanning 4.1x, from 0.0556% to 0.2288%.")
USERS.drop(columns=["n_imp", "n_eng", "n_pur"]).to_parquet(os.path.join(PATHS["artifacts"], "user_features.parquet"))
ITEMS.rename_axis("wid").reset_index().to_parquet(os.path.join(PATHS["artifacts"], "item_features.parquet"), index=False)
print("saved user_features.parquet and item_features.parquet")

,cells,lowest purchase rate,highest purchase rate,spread
0,27,0.0515%,0.2279%,4.4x


saved user_features.parquet and item_features.parquet


## 4. Pass 2: one feature row per impression

For each step-1 part this joins the user and item features to every candidate and adds the user-candidate features:

* **Comparison matches** (`int_shop_match`, `int_brand_match`, `int_cate4_match`, `int_name_sim`): the candidate shares a shop, brand, finest category or name terms with *anything* in the history, as submitted.
* **H2**: days since the user last touched the candidate's shop, brand or category through each interaction type, inside that type's window. Empty means no match.
* **H3**: days since the user clicked, carted or ordered this very item, and `ui_item_cart_no_order` for an item carted with no order since.
* **H4**: whether the user touched this item or its category under an earlier search sharing a term with the current one, and how long ago.
* **H5 and H6**: item popularity per type, exposure, and smoothed engagement and purchase rates from training users in the other folds.
* **H7**: popularity percentile within the slate and the share of the slate from the same shop.
* **H8**: share of the candidate's name terms found in recently clicked or carted items, and in ordered items.

While writing each part it also collects the evidence for section 5 from training rows: how often each match fires and the outcomes when it does, and each feature's within-slate AUC on a fixed sample of training slates.

In [ ]:
# 4.1 Within-slate AUC: does a feature rank engaged candidates above ignored ones inside the same slate?
AUC_USERS = np.sort(np.random.default_rng(SEED + 2).choice(train_ids, min(CFG["auc_users"], len(train_ids)), replace=False))

def slate_auc(d, feats):
    # Mann-Whitney AUC per slate, averaged over slates with at least one positive and one negative.
    # Oriented so higher should mean more relevant: 'days' features are negated, and a missing value
    # (no match) counts as the lowest.
    pos = (d["label"] > 0).to_numpy()
    uid = d["user_id"].to_numpy()
    n_pos = pd.Series(pos).groupby(uid).sum()
    n_neg = pd.Series(pos).groupby(uid).size() - n_pos
    ok = (n_pos > 0) & (n_neg > 0)
    out = {}
    for f in feats:
        x = d[f].to_numpy(dtype=float)
        x = np.where(np.isnan(x), -np.inf, -x if "days" in f else x)
        r = pd.Series(x).groupby(uid).rank(method="average").to_numpy()
        s_pos = pd.Series(r[pos]).groupby(uid[pos]).sum().reindex(n_pos.index, fill_value=0)
        auc = ((s_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))[ok]
        out[f] = (auc.sum(), auc.count())
    return out

In [ ]:
# 4.2 Pass 2: features per impression, written part by part
ATTRS = (("shop_id", "shop"), ("brand_id", "brand"), ("cate_id_4", "cate4"))
SRC = ("click", "cart", "order")
USER_COLS = [c for c in USERS.columns if c not in ("n_imp", "n_eng", "n_pur")]
lift_acc, auc_acc = {}, {}
base_acc = np.zeros(3)

def add_lift(name, flag, train, label):
    m = flag & train
    acc = lift_acc.setdefault(name, np.zeros(3))
    acc += (m.sum(), (m & (label > 0)).sum(), (m & (label == 3)).sum())

for old in glob.glob(os.path.join(PATHS["features"], "features_*.parquet")):
    os.remove(old)
with timer("pass 2"):
    for i, f in enumerate(parts("user_behavior")):
        t = pq.read_table(f, columns=["user_id", "query_terms", "candidate_wids", "candidate_labels", "history_wids",
                                      "history_types", "history_gaps", "history_age_days", "history_has_query",
                                      "history_qry_packed"])
        uid = t["user_id"].to_numpy().astype(np.int32)
        h = history_frame(t)
        h_ul = np.searchsorted(uid, h["user_id"].to_numpy())
        ha, _ = attrs(h["wid"].to_numpy())
        wids, n_c = flat(t, "candidate_wids")
        c_ul = np.repeat(np.arange(len(uid)), n_c)
        c = pd.DataFrame({"user_id": uid[c_ul], "wid": wids.astype(np.int64),
                          "label": flat(t, "candidate_labels")[0].astype(np.int8)})
        ca, _ = attrs(c["wid"].to_numpy())
        src_mask = {"click": ((h["type"] == CLICK) & (h["age"] < W_CLICK)).to_numpy(),
                    "cart": ((h["type"] == CART) & (h["age"] < W_CART)).to_numpy(),
                    "order": ((h["type"] == ORD) & (h["age"] < W_ORD)).to_numpy()}
        age, hw = h["age"].to_numpy(), h["wid"].to_numpy()

        for col, name in ATTRS:
            ok_h, ok_c = ha[col] >= 0, ca[col] >= 0
            # comparison: shared with anything in the history, any type, any age
            hk = np.unique(h_ul[ok_h].astype(np.int64) * T_KEY + ha[col][ok_h])
            c[f"int_{name}_match"] = (ok_c & np.isin(c_ul.astype(np.int64) * T_KEY + ca[col], hk)).astype(np.float32)
            # H2: days since last touched through each type, inside its window
            for src in SRC:
                m = src_mask[src] & ok_h
                days = min_age_lookup(h_ul[m], ha[col][m], age[m], c_ul, ca[col])
                c[f"ui_days_{name}_{src}"] = np.where(ok_c, days, np.nan).astype(np.float32)

        # H3: the item itself
        for src in SRC:
            m = src_mask[src]
            c[f"ui_item_days_{src}"] = min_age_lookup(h_ul[m], hw[m], age[m], c_ul, c["wid"].to_numpy())
        c["ui_item_cart_no_order"] = (c["ui_item_days_cart"].notna()
                                      & ~(c["ui_item_days_order"] <= c["ui_item_days_cart"])).astype(np.float32)

        # H4: the query as a key into history
        _, related, _ = query_links(t, h_ul)
        c["ui_item_days_related_query"] = min_age_lookup(h_ul[related], hw[related], age[related],
                                                         c_ul, c["wid"].to_numpy())
        m = related & (ha["cate_id_4"] >= 0)
        c["ui_cate4_days_related_query"] = np.where(ca["cate_id_4"] >= 0, min_age_lookup(
            h_ul[m], ha["cate_id_4"][m], age[m], c_ul, ca["cate_id_4"]), np.nan).astype(np.float32)

        # name overlap: comparison (whole history) and H8 (by source)
        if CFG["use_names"]:
            own, term = gather_terms(c["wid"].to_numpy())
            rt = np.unique(own.astype(np.int64) * T_KEY + term)              # distinct terms per candidate
            row, term_u = rt // T_KEY, rt % T_KEY
            n_terms = np.bincount(row, minlength=len(c))
            ukey = c_ul[row].astype(np.int64) * T_KEY + term_u

            def name_cov(mask):
                pairs = np.unique(h_ul[mask].astype(np.int64) * T_KEY + hw[mask])
                pu, pw = pairs // T_KEY, pairs % T_KEY
                o2, t2 = gather_terms(pw)
                hits = np.isin(ukey, np.unique(pu[o2] * T_KEY + t2))
                cov = np.bincount(row, weights=hits, minlength=len(c)) / np.maximum(n_terms, 1)
                return np.where(n_terms > 0, cov, np.nan).astype(np.float32)

            c["int_name_sim"] = name_cov(np.ones(len(h), bool))
            c["ui_name_cov_recent"] = name_cov(src_mask["click"] | src_mask["cart"])
            c["ui_name_cov_order"] = name_cov(src_mask["order"])

        # H5 and comparison popularity, H7 standing within the slate
        it = ITEMS.reindex(c["wid"].to_numpy())
        for col in ITEMS.columns:
            c[col] = it[col].fillna(0).to_numpy()
        g = c.groupby("user_id")
        c["s_pop_order_pct"] = g["i_pop_order"].rank(pct=True)
        c["s_pop_click_pct"] = g["i_pop_click"].rank(pct=True)
        shop = pd.Series(ca["shop_id"])
        c["s_shop_share"] = np.where(shop >= 0, shop.groupby([c_ul, shop]).transform("size") / n_c[c_ul], np.nan)

        # H6: label rates from training users in other folds
        fold = FOLD_OF.reindex(c["user_id"].to_numpy()).to_numpy()
        tot = LAB_TOT.reindex(c["wid"].to_numpy()).fillna(0).to_numpy()
        own_fold = LAB_FOLD.reindex(c["wid"].to_numpy() * 16 + np.maximum(fold, 0)).fillna(0).to_numpy()
        lab_counts = np.where((fold >= 0)[:, None], tot - own_fold, tot)        # imp, eng, pur from other users
        a = CFG["prior_strength"]
        c["i_label_imps_oof"] = lab_counts[:, 0]
        seen = lab_counts[:, 0] > 0                       # at least one other training user saw the item
        # change 2 (after the first run): no data stays empty instead of taking the overall rate
        c["i_eng_rate_oof"] = np.where(seen, (lab_counts[:, 1] + a * P_ENG) / (lab_counts[:, 0] + a), np.nan)
        c["i_pur_rate_oof"] = np.where(seen, (lab_counts[:, 2] + a * P_PUR) / (lab_counts[:, 0] + a), np.nan)

        # user features, identifiers, write
        c = c.join(USERS[USER_COLS], on="user_id")
        c["split"] = SPLIT_OF.reindex(c["user_id"].to_numpy()).to_numpy()
        c["fold"] = fold.astype(np.int8)
        feat_cols = [x for x in c.columns if x not in ("user_id", "wid", "label", "split", "fold")]
        c[feat_cols] = c[feat_cols].astype(np.float32)
        missing = [x for x in MODEL + COMPARISON if x not in c.columns]
        assert not missing, f"features declared in the model or comparison set but not built: {missing}"
        c.to_parquet(os.path.join(PATHS["features"], f"features_{i:02d}.parquet"), index=False)

        # evidence for section 5, training rows only
        train, label = (c["split"] == "train").to_numpy(), c["label"].to_numpy()
        base_acc += (train.sum(), (train & (label > 0)).sum(), (train & (label == 3)).sum())
        for col in c.columns:
            if col.startswith(("int_shop", "int_brand", "int_cate4", "ui_item_cart")):
                add_lift(col, c[col].to_numpy() > 0, train, label)
            elif col.startswith(("ui_days_", "ui_item_days_", "ui_cate4_days_")):
                add_lift(col, c[col].notna().to_numpy(), train, label)
            elif col in ("int_name_sim", "ui_name_cov_recent", "ui_name_cov_order"):
                add_lift(col, (c[col] >= 0.5).to_numpy(), train, label)
        sample = np.isin(c["user_id"].to_numpy(), AUC_USERS)
        for f_, (s_, n_) in slate_auc(c.loc[sample], [x for x in dict.fromkeys(MODEL + COMPARISON)]).items():
            auc_acc.setdefault(f_, np.zeros(2))[:] += (s_, n_)
        print(f"  features_{i:02d}: {len(c):,} rows x {len(feat_cols)} features")
        del t, h, c, it, tot, own_fold, lab_counts
FEATURES = feat_cols
print(f"{len(FEATURES)} features written to {PATHS['features']}")

  features_00: 1,777,726 rows x 49 features
  features_01: 1,785,821 rows x 49 features
  features_02: 1,781,631 rows x 49 features
  features_03: 1,778,446 rows x 49 features
  features_04: 1,789,049 rows x 49 features
  features_05: 1,782,074 rows x 49 features
  features_06: 1,784,465 rows x 49 features
  features_07: 1,792,313 rows x 49 features
  features_08: 1,238,487 rows x 49 features
[pass 2] 591.7s
49 features written to /content/drive/MyDrive/JDsearch_rawData/processed/artifacts/features


## 5. Evidence: what each feature separates

Three views, all on training users only:

* **5.1 User features** (as interim Table 10): purchase and engagement lift for the lowest and highest quartile of users, comparison versions next to H1. This measures differences *between users*: good evidence for the pooling problem, but not for ranking value, because these features are the same for every candidate in a slate.
* **5.2 Match features** (as interim Table 11): how often each fires and the engagement and purchase lift when it does. The interim measured these on 4,000 sampled rows; these use every training impression.
* **5.3 Within-slate AUC** for every feature, model and comparison: the evidence that matters for a re-ranker.

In [ ]:
# 5.1 User features: quartile lifts, comparison vs H1
tr = USERS.loc[USERS.index.isin(train_ids)]
base_pur, base_eng = tr["n_pur"].sum() / tr["n_imp"].sum(), tr["n_eng"].sum() / tr["n_imp"].sum()

def user_lift(feat):
    # Lift of the low and high group of users: quartiles for numeric features; for yes/no features
    # (change 3, after the first run) the users without vs with it, since quartiles of ties are arbitrary.
    x = tr[feat]
    has = x.notna()
    if x[has].nunique() <= 2:
        compares, lo_m, hi_m = "without vs with", has & (x == x[has].min()), has & (x == x[has].max())
    else:
        q = pd.Series(np.nan, index=tr.index)
        q[has] = pd.qcut(x[has].rank(method="first"), 4, labels=False)
        compares, lo_m, hi_m = "lowest vs highest quartile", q == 0, q == 3
    def rates(m):
        imp = tr.loc[m, "n_imp"].sum()
        return (tr.loc[m, "n_pur"].sum() / imp / base_pur, tr.loc[m, "n_eng"].sum() / imp / base_eng) if imp else (np.nan, np.nan)
    lo, hi, none = rates(lo_m), rates(hi_m), rates(~has)
    return {"compares": compares, "purchase lift: low / without": lo[0], "purchase lift: high / with": hi[0],
            "engagement lift: low / without": lo[1], "engagement lift: high / with": hi[1],
            "no value: share of users": (~has).mean(), "no value: purchase lift": none[0]}

user_feats = [f for f in COMPARISON if f in USER_COLS and f != "int_segment27"] + GROUPS["H1 activity inside its window"] + ["q_seen_before"]
lift_users = pd.DataFrame({f: user_lift(f) for f in user_feats}).T
lift_users.insert(0, "set", [GROUP_OF.get(f, "comparison") for f in lift_users.index])
show_result("User features: lift of the low and high group of users (training users)",
            lift_users.style.format("{:.2f}x", subset=[c for c in lift_users.columns if "lift" in c], na_rep="-")
                            .format("{:.1%}", subset=["no value: share of users"]),
            note=f"Base rates: purchase {base_pur:.3%}, engagement {base_eng:.3%} per impression.")

# Is the interim purchase share partly measuring "quiet lately"?
corr = tr[["int_purchase_share", "int_click_share", "int_history_length", "int_recency_log",
           "u_n_click_w", "u_days_since_click", "u_orders_per_cart_w"]].corr(method="spearman")
show_result("Spearman correlations between comparison and H1 user features (training users)",
            corr.round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

,set,compares,purchase lift: low / without,purchase lift: high / with,engagement lift: low / without,engagement lift: high / with,no value: share of users,no value: purchase lift
int_purchase_share,comparison,lowest vs highest quartile,0.44x,1.63x,1.04x,0.96x,0.0%,-
int_search_dependence,comparison,lowest vs highest quartile,0.73x,1.17x,1.00x,1.01x,0.0%,-
int_recency_log,comparison,lowest vs highest quartile,0.88x,1.15x,1.00x,0.98x,0.0%,-
int_history_length,comparison,lowest vs highest quartile,0.87x,1.10x,0.97x,1.06x,0.0%,-
int_order_count,comparison,lowest vs highest quartile,0.57x,1.48x,1.00x,1.01x,0.0%,-
int_click_share,comparison,lowest vs highest quartile,1.30x,0.60x,0.97x,1.05x,0.0%,-
int_cart_share,comparison,lowest vs highest quartile,0.99x,0.74x,1.03x,1.00x,0.0%,-
int_burstiness_log,comparison,lowest vs highest quartile,0.94x,1.03x,1.05x,0.95x,1.2%,0.68x
u_n_click_w,H1 activity inside its window,lowest vs highest quartile,1.15x,0.83x,0.97x,1.07x,0.0%,-
u_carts_per_click_w,H1 activity inside its window,lowest vs highest quartile,0.69x,1.30x,1.01x,0.97x,16.2%,1.25x


,int_purchase_share,int_click_share,int_history_length,int_recency_log,u_n_click_w,u_days_since_click,u_orders_per_cart_w
int_purchase_share,1.000000,-0.610000,0.010000,0.230000,-0.350000,0.130000,0.590000
int_click_share,-0.610000,1.000000,0.130000,-0.430000,0.680000,-0.180000,-0.170000
int_history_length,0.010000,0.130000,1.000000,-0.300000,0.750000,-0.110000,0.100000
int_recency_log,0.230000,-0.430000,-0.300000,1.000000,-0.450000,0.990000,0.050000
u_n_click_w,-0.350000,0.680000,0.750000,-0.450000,1.000000,-0.220000,-0.040000
u_days_since_click,0.130000,-0.180000,-0.110000,0.990000,-0.220000,1.000000,0.080000
u_orders_per_cart_w,0.590000,-0.170000,0.100000,0.050000,-0.040000,0.080000,1.000000


In [ ]:
# 5.2 Match features: how often they fire and the lift when they do (training impressions)
rows = []
for name, (n, eng, pur) in lift_acc.items():
    feat = name.split(" ")[0]
    rows.append({"feature": name + (" >= 0.5" if "name" in name else ""), "set": GROUP_OF.get(feat, "comparison"),
                 "fires on": n / base_acc[0],
                 "engagement lift": (eng / n) / (base_acc[1] / base_acc[0]) if n else np.nan,
                 "purchase lift": (pur / n) / (base_acc[2] / base_acc[0]) if n else np.nan})
lift_match = pd.DataFrame(rows).set_index("feature").sort_values(["set", "engagement lift"], ascending=[True, False])
show_result("Match features: share of impressions where each fires, and the lift when it does",
            lift_match.style.format("{:.2%}", subset=["fires on"]).format("{:.2f}x", subset=["engagement lift", "purchase lift"], na_rep="-"),
            note="Interim Table 11 (4,000 sampled rows, engagement): shop 3.04x, brand 1.44x, finest category 1.23x.")

,set,fires on,engagement lift,purchase lift
feature,,,,
ui_days_shop_click,H2 where a match comes from,4.50%,3.23x,2.93x
ui_days_shop_order,H2 where a match comes from,1.66%,3.07x,8.85x
ui_days_shop_cart,H2 where a match comes from,4.16%,3.05x,4.02x
ui_days_brand_click,H2 where a match comes from,14.33%,1.51x,1.16x
ui_days_brand_order,H2 where a match comes from,5.76%,1.44x,3.07x
ui_days_brand_cart,H2 where a match comes from,14.72%,1.38x,1.42x
ui_days_cate4_click,H2 where a match comes from,26.02%,1.28x,0.88x
ui_days_cate4_cart,H2 where a match comes from,24.06%,1.26x,1.09x
ui_days_cate4_order,H2 where a match comes from,10.37%,1.18x,2.10x


In [ ]:
# 5.3 Within-slate AUC for every feature (training slates with at least one positive and one negative)
auc = pd.DataFrame({"within-slate AUC": {f: s / n for f, (s, n) in auc_acc.items() if n},
                    "slates": {f: n for f, (s, n) in auc_acc.items()}})
auc["set"] = [GROUP_OF.get(f, "comparison") for f in auc.index]
auc["same for whole slate"] = [f in USER_COLS for f in auc.index]
auc = auc.sort_values(["set", "within-slate AUC"], ascending=[True, False])
show_result("Within-slate AUC: chance an engaged candidate scores higher than an ignored one in the same slate",
            auc[["set", "within-slate AUC", "same for whole slate", "slates"]].style
               .format({"within-slate AUC": "{:.3f}", "slates": "{:,.0f}"})
               .background_gradient(cmap="RdBu_r", subset=["within-slate AUC"], vmin=0.3, vmax=0.7),
            note="0.5 = no ranking signal on its own. Empty values count as lowest, so for H6 items nobody else saw rank last "
                 "and its AUC partly reflects exposure. Features marked as the same for the whole slate are 0.5 by construction; "
                 "the step-4 ablation tests whether they help through interactions.")

,set,within-slate AUC,same for whole slate,slates
u_n_click_w,H1 activity inside its window,0.500,True,"29,552"
u_carts_per_click_w,H1 activity inside its window,0.500,True,"29,552"
u_orders_per_cart_w,H1 activity inside its window,0.500,True,"29,552"
u_n_order_30d,H1 activity inside its window,0.500,True,"29,552"
u_n_order_w,H1 activity inside its window,0.500,True,"29,552"
u_days_since_click,H1 activity inside its window,0.500,True,"29,552"
u_days_since_order,H1 activity inside its window,0.500,True,"29,552"
ui_days_shop_click,H2 where a match comes from,0.557,False,"29,552"
ui_days_shop_cart,H2 where a match comes from,0.552,False,"29,552"
ui_days_brand_click,H2 where a match comes from,0.539,False,"29,552"


**Reading the results.**
* **H1 activity inside its window.** Every H1 feature is the same for all candidates in a slate (within-slate AUC 0.500), so on its own it cannot rank. It does separate users: the window-aware orders per cart gives a purchase lift of 0.39x to 1.70x from the lowest to the highest quartile, wider than the interim purchase share (0.44x to 1.63x). Prediction: H1 can only help a ranker through interactions with other features, so it is a candidate to fail the step-4 ablation.
* **H2 where a match comes from.** Supported. Recent-click matches carry more within-slate signal than order matches for shop (0.557 vs 0.521), brand (0.539 vs 0.515) and category (0.534 vs 0.510), while order matches carry the larger purchase lift (shop 8.85x vs 2.93x). Each single-source match scores below the pooled `int_` match (shop 0.584), because the pooled one fires more often (6.99% vs 4.50% of impressions); what H2 adds is which source the match comes from.
* **H3 the item itself.** Supported, but rare. An item the user ordered before has a 67.19x purchase lift and a carted one 20.61x, but they fire on 0.09% and 0.34% of impressions, so their AUC stays at 0.508 to 0.540.
* **H4 query as a key into history.** Weak. `q_seen_before` is slate-wide (0.500). The item matched through a related query has a 14.02x engagement lift but fires on 0.35% of impressions (AUC 0.528). Prediction: at risk in the ablation.
* **H5 popularity by type, H6 item conversion rate, H7 standing within the slate.** The strongest groups: exposure 0.763, cart popularity 0.758, the impressions behind the conversion rate 0.751 and the slate percentile of click popularity 0.753. The purchase rate is close to chance (0.555), because items no one has seen get the average rate, which sits above the near-zero rate of most shown items.
* **H8 name overlap by source.** Supported for recent activity (0.610), weak for orders (0.530).
* **The interim features.** Pooled popularity (`i_pop_any`, 0.777) is the strongest single feature, which is why it stays a comparison feature rather than a model feature. The interim user features are all slate-wide (0.500). The old recency is in effect days since the last click (Spearman 0.99). The interim purchase share is only weakly related to recency (0.23), so its lift is not a recency effect; it mostly tracks the window-aware orders per cart (0.59).


## 6. Feature dictionary and the fixed model set
Every feature with its role, hypothesis group, what it reads, and its evidence from section 5. Saved as `feature_dictionary.csv` for the report appendix. The model set and its groups are saved to `model_features.json`, which step 4 reads, so the set cannot drift.

In [ ]:
# 6. Feature dictionary
import re
from datetime import date
VERB = {"click": "clicked", "cart": "carted", "order": "ordered"}
NOUN = {"shop": "shop", "brand": "brand", "cate4": "finest category"}
FIXED = {
    "int_history_length": "Number of history events, all types and ages",
    "int_order_count": "Number of orders in the whole history",
    "int_purchase_share": "Orders / all history events",
    "int_click_share": "Clicks / all history events",
    "int_cart_share": "Carts / all history events",
    "int_search_dependence": "Share of history events with a search behind them",
    "int_recency_log": "Log seconds from the last history event to the test query",
    "int_burstiness_log": "Log median seconds between consecutive history events (our reconstruction)",
    "int_segment27": "Interim segment: thirds of purchase share x recency x search dependence",
    "int_name_sim": "Share of the candidate's name terms found in any history item's name (our reconstruction)",
    "i_exposure_n": "Number of slates the item appears in",
    "i_pop_any": "Users whose history holds the item, any type or age",
    "i_eng_rate_oof": "Smoothed engagement rate of the item among training users in other folds (empty if none saw it)",
    "i_pur_rate_oof": "Smoothed purchase rate of the item among training users in other folds (empty if none saw it)",
    "i_label_imps_oof": "Training impressions behind those rates (how much to trust them)",
    "ui_item_cart_no_order": "Item carted inside the cart window with no order since: a pending purchase",
    "ui_item_days_related_query": "Days since the user touched this item under a search sharing a term with the current one",
    "ui_cate4_days_related_query": "Days since the user touched this finest category under a related search",
    "ui_name_cov_recent": "Share of the candidate's name terms found in recently clicked or carted items",
    "ui_name_cov_order": "Share of the candidate's name terms found in ordered items",
    "q_seen_before": "The user ran exactly this query before",
    "u_n_click_w": "Clicks inside the click window",
    "u_n_order_30d": "Orders in the last 30 days",
    "u_n_order_w": "Orders inside the order window",
    "u_carts_per_click_w": "Carts per click, both counted inside the click window",
    "u_orders_per_cart_w": "Orders per cart, both counted inside the cart window",
    "u_days_since_click": "Days since the last click (empty: none inside the click window)",
    "u_days_since_order": "Days since the last order (empty: none inside the order window)",
    "s_pop_order_pct": "Order popularity percentile within the slate",
    "s_pop_click_pct": "Click popularity percentile within the slate",
    "s_shop_share": "Share of the slate from the candidate's shop",
}

def describe(f):
    m = re.match(r"(?:int_(shop|brand|cate4)_match)$|(?:ui_days_(shop|brand|cate4)_(click|cart|order))$|"
                 r"(?:ui_item_days_(click|cart|order))$|(?:i_pop_(click|cart|order))$", f)
    if f in FIXED:
        what = FIXED[f]
    elif m and m.group(1):
        what = f"Candidate's {NOUN[m.group(1)]} appears anywhere in the history"
    elif m and m.group(2):
        what = f"Days since the user last {VERB[m.group(3)]} an item from the candidate's {NOUN[m.group(2)]}"
    elif m and m.group(4):
        what = f"Days since the user last {VERB[m.group(4)]} this item"
    elif m and m.group(5):
        what = f"Users who {VERB[m.group(5)]} the item inside its window"
    else:
        what = ""
    in_model, in_comp = f in GROUP_OF, f in COMPARISON
    role = "model and comparison" if in_model and in_comp else "model" if in_model else "comparison" if in_comp else "unused"
    if f.startswith(("i_eng", "i_pur", "i_label")):
        reads = "labels of training users, other folds only"
    elif f.startswith(("q_", "ui_item_days_related", "ui_cate4_days_related")):
        reads = "current query, history queries" + (", product metadata" if "cate4" in f else "")
    elif f == "i_exposure_n":
        reads = "candidate lists (no labels)"
    elif f == "s_shop_share":
        reads = "candidate lists, product metadata"
    elif f.startswith(("i_", "s_")):
        reads = "all users' history (no labels)"
    else:
        reads = "history" + (", product metadata" if any(k in f for k in ("shop", "brand", "cate4", "name")) else "")
    window = ("whole history pooled" if f.startswith("int_") or f == "i_pop_any" else
              f"clicks {W_CLICK:g}d, carts {W_CART:g}d, orders {W_ORD:g}d"
              if f.startswith(("u_", "ui_days", "ui_item_days_click", "ui_item_days_cart", "ui_item_days_order",
                               "ui_item_cart", "ui_name", "i_pop_c", "i_pop_o")) else "-")
    group = GROUP_OF.get(f, "")
    ev = []
    if f in auc.index and not auc.loc[f, "same for whole slate"]:
        ev.append(f"within-slate AUC {auc.loc[f, 'within-slate AUC']:.3f}")
    if f in lift_match.index or f + " >= 0.5" in lift_match.index:
        r = lift_match.loc[f if f in lift_match.index else f + " >= 0.5"]
        ev.append(f"fires on {r['fires on']:.1%}, engagement lift {r['engagement lift']:.2f}x")
    if f in lift_users.index:
        r = lift_users.loc[f]
        ev.append(f"purchase lift {r['purchase lift: low / without']:.2f}x to {r['purchase lift: high / with']:.2f}x ({r['compares']})")
    return {"role": role, "group": group, "hypothesis": HYPOTHESIS.get(group, "reproduces the interim plan" if in_comp else ""),
            "what it measures": what, "reads": reads, "history window": window, "evidence (section 5)": "; ".join(ev)}

dictionary = pd.DataFrame({f: describe(f) for f in dict.fromkeys(MODEL + COMPARISON)}).T.rename_axis("feature")
dictionary.to_csv(os.path.join(PATHS["artifacts"], "feature_dictionary.csv"))
with open(os.path.join(PATHS["artifacts"], "model_features.json"), "w") as f:
    json.dump({"version": "1.0", "status": "frozen before step 4: no features added or removed after this point",
               "model_features": MODEL, "groups": GROUPS, "hypotheses": HYPOTHESIS,
               "interim_baseline": INTERIM_BASELINE, "comparison": COMPARISON,
               "exploratory_variants": {"+ pooled popularity": {
                   "add": ["i_pop_any"],
                   "note": "Found after looking at section 5 (highest within-slate AUC on training users). "
                           "Tested on validation in step 4 and reported separately from the main model."}},
               "history_windows_days": WIN, "created": str(date.today())}, f, indent=2)
show_result(f"Frozen model set v1.0: {len(MODEL)} features in {len(GROUPS)} hypothesis groups; comparison set: {len(COMPARISON)} features",
            dictionary.groupby(["role", "group"]).size().to_frame("features"))
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 140):
    display(dictionary)

features
role                 group                                   
comparison                                                 14
model                H1 activity inside its window          7
                     H2 where a match comes from            9
                     H3 the item itself                     4
                     H4 query as a key into history         3
                     H5 popularity by type                  3
                     H6 item conversion rate                2
                     H7 standing within the slate           3
                     H8 name overlap by source              2
model and comparison H5 popularity by type                  1
                     H6 item conversion rate                1

,role,group,hypothesis,what it measures,reads,history window,evidence (section 5)
feature,,,,,,,
u_n_click_w,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,Clicks inside the click window,history,"clicks 7d, carts 90d, orders 365d",purchase lift 1.15x to 0.83x (lowest vs highest quartile)
u_carts_per_click_w,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,"Carts per click, both counted inside the click window",history,"clicks 7d, carts 90d, orders 365d",purchase lift 0.69x to 1.30x (lowest vs highest quartile)
u_orders_per_cart_w,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,"Orders per cart, both counted inside the cart window",history,"clicks 7d, carts 90d, orders 365d",purchase lift 0.39x to 1.70x (lowest vs highest quartile)
u_n_order_30d,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,Orders in the last 30 days,history,"clicks 7d, carts 90d, orders 365d",purchase lift 0.56x to 1.49x (lowest vs highest quartile)
u_n_order_w,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,Orders inside the order window,history,"clicks 7d, carts 90d, orders 365d",purchase lift 0.57x to 1.48x (lowest vs highest quartile)
u_days_since_click,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,Days since the last click (empty: none inside the click window),history,"clicks 7d, carts 90d, orders 365d",purchase lift 0.80x to 1.09x (lowest vs highest quartile)
u_days_since_order,model,H1 activity inside its window,Activity describes intent and habit only inside the period each type is recorded,Days since the last order (empty: none inside the order window),history,"clicks 7d, carts 90d, orders 365d",purchase lift 1.44x to 0.63x (lowest vs highest quartile)
ui_days_shop_click,model,H2 where a match comes from,"Recent clicks signal intent, carts consideration, orders loyalty; pooled, these mix",Days since the user last clicked an item from the candidate's shop,"history, product metadata","clicks 7d, carts 90d, orders 365d","within-slate AUC 0.557; fires on 4.5%, engagement lift 3.23x"
ui_days_shop_cart,model,H2 where a match comes from,"Recent clicks signal intent, carts consideration, orders loyalty; pooled, these mix",Days since the user last carted an item from the candidate's shop,"history, product metadata","clicks 7d, carts 90d, orders 365d","within-slate AUC 0.552; fires on 4.2%, engagement lift 3.05x"


## 7. Outputs of this notebook

In [ ]:
for folder in (PATHS["features"], PATHS["artifacts"]):
    for f in sorted(glob.glob(os.path.join(folder, "*.*"))):
        print(f"{os.path.relpath(f, BASE):<60} {os.path.getsize(f) / 1e6:>9,.1f} MB")

processed/artifacts/features/features_00.parquet                  39.1 MB
processed/artifacts/features/features_01.parquet                  39.5 MB
processed/artifacts/features/features_02.parquet                  39.4 MB
processed/artifacts/features/features_03.parquet                  39.2 MB
processed/artifacts/features/features_04.parquet                  39.5 MB
processed/artifacts/features/features_05.parquet                  39.3 MB
processed/artifacts/features/features_06.parquet                  39.3 MB
processed/artifacts/features/features_07.parquet                  39.6 MB
processed/artifacts/features/features_08.parquet                  27.4 MB
processed/artifacts/feature_dictionary.csv                         0.0 MB
processed/artifacts/folds.parquet                                  0.9 MB
processed/artifacts/history_windows.json                           0.0 MB
processed/artifacts/item_features.parquet                         82.1 MB
processed/artifacts/model_features.jso

## 8. AI interaction log
*Record essential exchanges as you go; they are merged into the report appendix.*

| # | Core prompt and task objective | AI output summary | Critical evaluation | Refinement and final action |
|---|---|---|---|---|
| 1 | "Using our interim features and the prof's comments, how should our feature set change, and what are we missing?" | It explained why lumping clicks, carts and orders together mixes time periods of very different lengths. It also showed that our planned interaction rate was leaking the answer: 75% of items appear in only one slate, so their rate is just their own label. Computing it from other users only fixes that. It then built three groups: our interim features, window-aware versions and new ones | We ended up with 64 features piled together, which goes against our own interim's rule of testing each feature before keeping it | This led to the next two questions. We also learned a feature can leak the answer through an average, not just by copying the label |
| 2 | "Should we be building a properly justified feature set first and then training the model, instead of throwing everything in?" | It agreed and explained how: write the hypotheses down before seeing any results, use one measure that fits a ranking task (within-slate AUC), and don't pick features by trial and error, because that just fits the validation set | There's a real tension between exploring features and keeping the final test honest. We also noticed three of our "as submitted" features had to be reconstructed, because the interim never defined them precisely | We settled on 35 model features in 8 hypothesis groups, kept 16 interim features for comparison, dropped 11 with reasons, and froze the set in model_features.json. The reconstructed features are labelled, and step 4 tests each group |
| 3 | "Here are our results. How are our features doing, and how should we read them?" | It went through the evidence one hypothesis at a time. The old pooled features were mostly measuring how recently someone browsed (old recency vs days since last click: 0.99). It also explained why the purchase rate scored 0.331: unseen items got the average rate, which sat above the near-zero rate of popular items | A wrong prediction is still a result. H3 said a past order might mean the need was already met, but past orders actually came with a 67x repurchase lift, so we report it as refuted rather than quietly rewording it. We also saw how smoothing goes wrong on sparse data | This led to the next question |
| 4 | "So we can't tell yet whether all the features help. Does that mean we can't add new ones?" | It explained when a new feature is fine to add: a new idea with its own reason can go in before step 4; anything suggested by results is exploratory; nothing gets added because of validation results | We took that as our rule for the rest of the project | We added nothing new. Step 4 tested pooled popularity as an exploratory extra (-0.0007 NDCG@10) and left it out, as the rule said |